# Automated FinTech Portfolio Analysis Project

This project fetches stock market data using Python and prepares data for portfolio analysis and dashboard visualization.

!pip install yfinance  #yfinance is a finance data specific python library that will fetch data from yahoo finance site

In [1]:
import yfinance as yf 
import pandas as pd

In [2]:
stock_dictionary = {
    "Finance": ["JPM", "MS", "GS", "BAC", "C"],
    "Technology": ["AAPL", "MSFT", "AMZN", "GOOG", "NVDA"],
    "Healthcare": ["PFE", "JNJ", "MRK", "UNH", "ABBV"],
    "Automobile": ["TSLA", "F", "GM", "TM", "RIVN","TATAMOTORS.NS"]
}
tickers = []

for stocks in stock_dictionary.values():
    tickers.extend(stocks)

try:
    portfolio_data = yf.download(tickers, period="6mo")

except Exception as e:
    print("Download failed")
    print(e)

[**********************90%******************     ]  19 of 21 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  20 of 21 completed

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")


In [3]:
portfolio_data.shape

(124, 106)

In [4]:
portfolio_data.head(5)

Price          Adj Close       Close                                     \
Ticker     TATAMOTORS.NS        AAPL        ABBV        AMZN        BAC   
Date                                                                      
2025-12-26           NaN  272.893005  226.267258  232.520004  55.565536   
2025-12-29           NaN  273.252350  227.113373  232.070007  54.754364   
2025-12-30           NaN  272.573547  226.031143  232.529999  54.685120   
2025-12-31           NaN  271.355835  224.801315  230.820007  54.408131   
2026-01-02           NaN  270.507446  225.608078  226.500000  55.347912   

Price                                                                 ...  \
Ticker               C          F         GM        GOOG          GS  ...   
Date                                                                  ...   
2025-12-26  119.231567  13.003484  82.684563  314.548950  898.332275  ...   
2025-12-29  116.964165  12.974174  82.555153  313.979736  883.614990  ...   
2025-12-30  116.053246  12.925325  81.957863  314.139496  875.929443  ...   
2025-12-31  115.538383  12.817859  80.952431  313.390472  870.561523  ...   
2026-01-02  117.528542  13.032792  80.613968  314.908508  905.562317  ...   

Price        Volume                                                    \
Ticker          MRK       MS      MSFT       NVDA       PFE      RIVN   
Date                                                                    
2025-12-26  6280700  2564700   8842200  139740300  21617800  20503000   
2025-12-29  8086900  2464300  10893400  120006100  33067700  20707600   
2025-12-30  6497900  2365500  13944500   97687300  28821700  38607700   
2025-12-31  7557300  3098100  15601600  120100500  29409200  21573800   
2026-01-02  9992400  4509900  25571600  148240500  35976700  42740200   

Price                                                
Ticker     TATAMOTORS.NS      TM      TSLA      UNH  
Date                                                 
2025-12-26           NaN  146000  58780700  4359300  
2025-12-29           NaN  247100  66263000  4346800  
2025-12-30           NaN  184500  59238500  4432500  
2025-12-31           NaN  151700  49078000  4285200  
2026-01-02           NaN  277800  85535400  6863600  

[5 rows x 106 columns]

In [5]:
# data quality checks
failed_tickers = portfolio_data["Close"].isna().all()
failed_tickers = failed_tickers[failed_tickers]
failed_ticker_list = failed_tickers.index.tolist()
print("Failed tickers:")
print(failed_ticker_list)

Failed tickers:
['TATAMOTORS.NS']


In [6]:
portfolio_data = portfolio_data.drop(
    columns=failed_ticker_list,
    level=1
)

#### Clean dataframe presentation

In [7]:
latest_date = portfolio_data.index[-1]

In [8]:
clean_table = pd.DataFrame({
    "Close": portfolio_data["Close"].loc[latest_date],
    "High": portfolio_data["High"].loc[latest_date],
    "Low": portfolio_data["Low"].loc[latest_date],
    "Open": portfolio_data["Open"].loc[latest_date],
    "Volume": portfolio_data["Volume"].loc[latest_date]
})

In [9]:
clean_table.head()

,Close,High,Low,Open,Volume
Ticker,,,,,
AAPL,275.149994,288.799988,273.750000,287.399994,107013700
ABBV,243.139999,244.610001,233.660004,233.660004,10278100
AMZN,227.009995,232.320007,225.550003,232.020004,77674100
BAC,58.189999,59.200001,57.849998,57.849998,33747400
C,144.979996,147.789993,144.820007,144.889999,12155900


In [10]:
# need date column as well
clean_table = clean_table.reset_index()

In [11]:
clean_table.head(1)

,Ticker,Close,High,Low,Open,Volume
0,AAPL,275.149994,288.799988,273.75,287.399994,107013700


In [12]:
# add date column
clean_table["Date"] = latest_date

In [13]:
# stock data of lastest date
clean_portfolio_data = clean_table[["Ticker","Date","Close","High","Low","Open","Volume"]]
clean_portfolio_data.head()

,Ticker,Date,Close,High,Low,Open,Volume
0,AAPL,2026-06-25,275.149994,288.799988,273.750000,287.399994,107013700
1,ABBV,2026-06-25,243.139999,244.610001,233.660004,233.660004,10278100
2,AMZN,2026-06-25,227.009995,232.320007,225.550003,232.020004,77674100
3,BAC,2026-06-25,58.189999,59.200001,57.849998,57.849998,33747400
4,C,2026-06-25,144.979996,147.789993,144.820007,144.889999,12155900


In [14]:
# map sectors of respective stocks 
ticker_sector = {}

for sector, stocks in stock_dictionary.items():
    for stock in stocks:
        ticker_sector[stock] = sector

In [15]:
clean_portfolio_data["Sector"] = clean_portfolio_data["Ticker"].map(ticker_sector)

C:\Users\Ashlesha\AppData\Local\Temp\ipykernel_15512\1232821456.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_portfolio_data["Sector"] = clean_portfolio_data["Ticker"].map(ticker_sector)


In [16]:
clean_portfolio_data = clean_portfolio_data[["Ticker", "Sector", "Date", "Close", "High","Low", "Open", "Volume"]]

clean_portfolio_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume
0,AAPL,Technology,2026-06-25,275.149994,288.799988,273.750000,287.399994,107013700
1,ABBV,Healthcare,2026-06-25,243.139999,244.610001,233.660004,233.660004,10278100
2,AMZN,Technology,2026-06-25,227.009995,232.320007,225.550003,232.020004,77674100
3,BAC,Finance,2026-06-25,58.189999,59.200001,57.849998,57.849998,33747400
4,C,Finance,2026-06-25,144.979996,147.789993,144.820007,144.889999,12155900


In [17]:
clean_portfolio_data.isnull().sum()

Ticker    0
Sector    0
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [18]:
clean_portfolio_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Ticker  20 non-null     object        
 1   Sector  20 non-null     object        
 2   Date    20 non-null     datetime64[ns]
 3   Close   20 non-null     float64       
 4   High    20 non-null     float64       
 5   Low     20 non-null     float64       
 6   Open    20 non-null     float64       
 7   Volume  20 non-null     int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(2)
memory usage: 1.4+ KB


In [19]:
clean_portfolio_data['Date'] = pd.to_datetime(clean_portfolio_data['Date'],format='%d-%m-%Y')

In [20]:
clean_portfolio_data.describe()

,Date,Close,High,Low,Open,Volume
count,20,20.000000,20.000000,20.000000,20.000000,2.000000e+01
mean,2026-06-25 00:00:00,245.956496,251.527000,243.334001,247.653999,3.585530e+07
min,2026-06-25 00:00:00,14.110000,14.360000,13.890000,13.920000,5.774000e+05
25%,2026-06-25 00:00:00,113.719997,114.577499,110.347500,110.997498,9.644875e+06
50%,2026-06-25 00:00:00,224.024994,229.900002,223.200005,227.505005,2.035520e+07
75%,2026-06-25 00:00:00,336.887497,343.255005,335.045006,335.449997,5.882998e+07
max,2026-06-25 00:00:00,1065.089966,1105.250000,1062.099976,1095.619995,1.495500e+08
std,NaN,229.450465,237.393614,228.450940,235.147576,3.973149e+07


In [21]:
# check duplicates
clean_portfolio_data.duplicated().sum()

0

In [22]:
# clean dataset for the latest date
portfolio_data_with_sectors_lastestDay = clean_portfolio_data.copy()

In [23]:
portfolio_data.head(2)

Price            Close                                                 \
Ticker            AAPL        ABBV        AMZN        BAC           C   
Date                                                                    
2025-12-26  272.893005  226.267258  232.520004  55.565536  119.231567   
2025-12-29  273.252350  227.113373  232.070007  54.754364  116.964165   

Price                                                                 ...  \
Ticker              F         GM        GOOG          GS         JNJ  ...   
Date                                                                  ...   
2025-12-26  13.003484  82.684563  314.548950  898.332275  205.351059  ...   
2025-12-29  12.974174  82.555153  313.979736  883.614990  205.281830  ...   

Price        Volume                                                   \
Ticker          JPM      MRK       MS      MSFT       NVDA       PFE   
Date                                                                   
2025-12-26  4158300  6280700  2564700   8842200  139740300  21617800   
2025-12-29  8635300  8086900  2464300  10893400  120006100  33067700   

Price                                            
Ticker          RIVN      TM      TSLA      UNH  
Date                                             
2025-12-26  20503000  146000  58780700  4359300  
2025-12-29  20707600  247100  66263000  4346800  

[2 rows x 100 columns]

In [24]:
historical_stock_data = portfolio_data.copy()

In [25]:
historical_stock_data = (historical_stock_data.stack(level=1).reset_index())

historical_stock_data.head()

Price,Date,Ticker,Close,High,Low,Open,Volume
0,2025-12-26,AAPL,272.893005,274.859353,272.353998,273.651606,21521800
1,2025-12-26,ABBV,226.267258,226.887092,224.988242,226.109852,1593900
2,2025-12-26,AMZN,232.520004,232.990005,231.179993,232.039993,15994700
3,2025-12-26,BAC,55.565536,55.941448,55.427044,55.674353,15259400
4,2025-12-26,C,119.231567,120.835582,118.488969,120.449432,10603700


In [26]:
historical_stock_data.columns

Index(['Date', 'Ticker', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

In [27]:
# map sectors of respective stocks 
ticker_sector = {}

for sector, stocks in stock_dictionary.items():
    for stock in stocks:
        ticker_sector[stock] = sector

In [28]:
historical_stock_data["Sector"] = historical_stock_data["Ticker"].map(ticker_sector)

In [29]:
historical_stock_data = historical_stock_data[["Ticker", "Sector", "Date", "Close", "High","Low", "Open", "Volume"]]

In [30]:
historical_stock_data.head(2)

Price,Ticker,Sector,Date,Close,High,Low,Open,Volume
0,AAPL,Technology,2025-12-26,272.893005,274.859353,272.353998,273.651606,21521800
1,ABBV,Healthcare,2025-12-26,226.267258,226.887092,224.988242,226.109852,1593900


In [31]:
historical_stock_data.isnull().sum()

Price
Ticker    0
Sector    0
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [32]:
historical_stock_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2480 entries, 0 to 2479
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Ticker  2480 non-null   object        
 1   Sector  2480 non-null   object        
 2   Date    2480 non-null   datetime64[ns]
 3   Close   2480 non-null   float64       
 4   High    2480 non-null   float64       
 5   Low     2480 non-null   float64       
 6   Open    2480 non-null   float64       
 7   Volume  2480 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(2)
memory usage: 155.1+ KB


In [33]:
historical_stock_data['Date'] = pd.to_datetime(historical_stock_data['Date'],format='%d-%m-%Y')

In [34]:
historical_stock_data.describe()

Price,Date,Close,High,Low,Open,Volume
count,2480,2480.000000,2480.000000,2480.000000,2480.000000,2.480000e+03
mean,2026-03-27 10:27:05.806451712,233.998190,236.905180,231.013661,233.931328,3.134808e+07
min,2025-12-26 00:00:00,11.070457,11.307469,10.971701,11.218589,1.460000e+05
25%,2026-02-10 18:00:00,99.302450,100.231952,97.380884,99.418780,7.248475e+06
50%,2026-03-26 12:00:00,212.724998,215.029999,209.864614,211.901466,1.687300e+07
75%,2026-05-11 06:00:00,305.671967,308.970541,301.725189,305.414008,4.189890e+07
max,2026-06-25 00:00:00,1106.369995,1125.000000,1094.290039,1120.000000,3.608079e+08
std,NaN,201.065725,203.765546,198.217474,200.952835,3.971415e+07


In [35]:
historical_stock_data.duplicated().sum()

0

In [36]:
# cleanups
historical_stock_data.columns.name = None
historical_stock_data = historical_stock_data.sort_values(["Ticker", "Date"])

In [37]:
historical_stock_data["Daily Return %"] = round(historical_stock_data.groupby('Ticker')['Close'].pct_change()*100,2)

In [38]:
historical_stock_data.head(2)

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
0,AAPL,Technology,2025-12-26,272.893005,274.859353,272.353998,273.651606,21521800,NaN
20,AAPL,Technology,2025-12-29,273.252350,273.851213,271.844961,272.184327,23715200,0.13


In [39]:
historical_stock_data["Daily Return %"].describe()

count    2460.000000
mean        0.033366
std         2.152991
min       -19.610000
25%        -1.090000
50%        -0.010000
75%         1.140000
max        26.640000
Name: Daily Return %, dtype: float64

In [40]:
cumulative_returns = historical_stock_data.groupby('Ticker').agg(
                                                                 First_close = ('Close','first'),
                                                                 Last_close = ('Close','last'))
cumulative_returns

,First_close,Last_close
Ticker,,
AAPL,272.893005,275.149994
ABBV,226.267258,243.139999
AMZN,232.520004,227.009995
BAC,55.565536,58.189999
C,119.231567,144.979996
F,13.003484,14.110000
GM,82.684563,78.529999
GOOG,314.548950,342.190002
GS,898.332275,1065.089966


In [41]:
cumulative_returns["Cumulative Return %"] = round((cumulative_returns['Last_close'] - cumulative_returns['First_close']) / cumulative_returns['First_close'] *100 , 2)

cumulative_returns = cumulative_returns.sort_values("Cumulative Return %",ascending=False)
cumulative_returns.head()

,First_close,Last_close,Cumulative Return %
Ticker,,,
UNH,327.400208,415.529999,26.92
MS,179.906097,221.039993,22.86
C,119.231567,144.979996,21.60
JNJ,205.351059,244.880005,19.25
MRK,105.238129,125.449997,19.21


In [42]:
# average trading volume
cumulative_returns["Total Volume"] = (historical_stock_data.groupby("Ticker")["Volume"].sum())

# sector mapping
sector_map = (historical_stock_data.groupby("Ticker")["Sector"].first())
cumulative_returns["Sector"] = sector_map

In [43]:
stock_performance_summary = (cumulative_returns.reset_index())

stock_performance_summary = stock_performance_summary[["Ticker","Sector","First_close","Last_close","Cumulative Return %","Total Volume"]]
stock_performance_summary.head()

,Ticker,Sector,First_close,Last_close,Cumulative Return %,Total Volume
0,UNH,Healthcare,327.400208,415.529999,26.92,1069496700
1,MS,Finance,179.906097,221.039993,22.86,821587900
2,C,Finance,119.231567,144.979996,21.60,1676869800
3,JNJ,Healthcare,205.351059,244.880005,19.25,1027516000
4,MRK,Healthcare,105.238129,125.449997,19.21,1366032700


In [44]:
# investor's porfolios 

investor_profiles = { 
    "Investor_A": {"capital": 10000, "weights": {"JPM": 0.30, "BAC": 0.30, "ABBV": 0.20, "JNJ": 0.20}},
    "Investor_B": { "capital": 10000,"weights": {"NVDA": 0.30, "TSLA": 0.25, "AMZN": 0.25, "AAPL": 0.20}}
}
investor_profiles

{'Investor_A': {'capital': 10000,
  'weights': {'JPM': 0.3, 'BAC': 0.3, 'ABBV': 0.2, 'JNJ': 0.2}},
 'Investor_B': {'capital': 10000,
  'weights': {'NVDA': 0.3, 'TSLA': 0.25, 'AMZN': 0.25, 'AAPL': 0.2}}}

In [45]:
for investor, details in investor_profiles.items():

    total_weight = sum(details["weights"].values())

    if total_weight != 1:
        raise ValueError(
            f"{investor} weights do not sum to 1. Current total: {total_weight}"
        )

-  validated that portfolio allocations summed to 100%, if not, the pipeline stops and raises an error to prevent incorrect return calculations

#### now lets calculate weighted return calculation for each investor

##### Investor A portfolio summary

In [46]:
investor_A_stocks = investor_profiles['Investor_A']['weights'].keys()
investor_A_stocks

dict_keys(['JPM', 'BAC', 'ABBV', 'JNJ'])

In [47]:
# filter the complete stock historical data to just gets stocks data held by the investor
investor_A_data = historical_stock_data[historical_stock_data['Ticker'].isin(investor_A_stocks)].copy()
investor_A_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
1,ABBV,Healthcare,2025-12-26,226.267258,226.887092,224.988242,226.109852,1593900,NaN
21,ABBV,Healthcare,2025-12-29,227.113373,228.274331,226.031140,226.434510,3803000,0.37
41,ABBV,Healthcare,2025-12-30,226.031143,227.477413,224.929214,227.044514,2348000,-0.48
61,ABBV,Healthcare,2025-12-31,224.801315,226.532897,224.742271,226.031136,3196900,-0.54
81,ABBV,Healthcare,2026-01-02,225.608078,227.015000,222.066204,225.047288,3336800,0.36


In [48]:
investor_A_data['Weight'] = investor_A_data['Ticker'].map(investor_profiles['Investor_A']['weights'])

In [49]:
investor_A_data.head(2)

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %,Weight
1,ABBV,Healthcare,2025-12-26,226.267258,226.887092,224.988242,226.109852,1593900,NaN,0.2
21,ABBV,Healthcare,2025-12-29,227.113373,228.274331,226.031140,226.434510,3803000,0.37,0.2


In [50]:
investor_A_data["Weighted Return"] = (investor_A_data["Daily Return %"] * investor_A_data["Weight"])

In [51]:
portfolio_A_daily_returns = (investor_A_data.groupby("Date")["Weighted Return"].sum().reset_index())
portfolio_A_daily_returns.rename(columns={"Weighted Return": "Portfolio Daily Return %"},inplace = True )

In [52]:
portfolio_A_daily_returns.head()

,Date,Portfolio Daily Return %
0,2025-12-26,0.000
1,2025-12-29,-0.751
2,2025-12-30,-0.227
3,2025-12-31,-0.368
4,2026-01-02,0.932


In [53]:
initial_capital = investor_profiles["Investor_A"]["capital"]

portfolio_A_daily_returns["Portfolio Value"] = round((initial_capital * 
                                                (1 + portfolio_A_daily_returns["Portfolio Daily Return %"] / 100).cumprod()),3)
portfolio_A_daily_returns.head()

,Date,Portfolio Daily Return %,Portfolio Value
0,2025-12-26,0.000,10000.00
1,2025-12-29,-0.751,9924.90
2,2025-12-30,-0.227,9902.37
3,2025-12-31,-0.368,9865.93
4,2026-01-02,0.932,9957.88


In [54]:
portfolio_A_daily_returns["Portfolio Cumulative Return %"] = round(((portfolio_A_daily_returns["Portfolio Value"] - initial_capital) / initial_capital )*100,3)
portfolio_A_daily_returns.head(3)

,Date,Portfolio Daily Return %,Portfolio Value,Portfolio Cumulative Return %
0,2025-12-26,0.000,10000.00,0.000
1,2025-12-29,-0.751,9924.90,-0.751
2,2025-12-30,-0.227,9902.37,-0.976


##### Investor B portfolio summary

In [55]:
investor_B_stocks = investor_profiles['Investor_B']['weights'].keys()
investor_B_stocks

dict_keys(['NVDA', 'TSLA', 'AMZN', 'AAPL'])

In [56]:
# filter the complete stock historical data to just gets stocks data held by the investor
investor_B_data = historical_stock_data[historical_stock_data['Ticker'].isin(investor_B_stocks)].copy()
investor_B_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
0,AAPL,Technology,2025-12-26,272.893005,274.859353,272.353998,273.651606,21521800,NaN
20,AAPL,Technology,2025-12-29,273.252350,273.851213,271.844961,272.184327,23715200,0.13
40,AAPL,Technology,2025-12-30,272.573547,273.571693,271.775043,272.304059,22139600,-0.25
60,AAPL,Technology,2025-12-31,271.355835,273.172467,271.246054,272.553622,27293600,-0.45
80,AAPL,Technology,2026-01-02,270.507446,277.324767,268.501164,271.755128,37838100,-0.31


In [57]:
investor_B_data['Weight'] = investor_B_data['Ticker'].map(investor_profiles['Investor_B']['weights'])

In [58]:
investor_B_data.head(2)

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %,Weight
0,AAPL,Technology,2025-12-26,272.893005,274.859353,272.353998,273.651606,21521800,NaN,0.2
20,AAPL,Technology,2025-12-29,273.252350,273.851213,271.844961,272.184327,23715200,0.13,0.2


In [59]:
investor_B_data["Weighted Return"] = (investor_B_data["Daily Return %"] * investor_B_data["Weight"])

In [60]:
portfolio_B_daily_returns = (investor_B_data.groupby("Date")["Weighted Return"].sum().reset_index())
portfolio_B_daily_returns.rename(columns={"Weighted Return": "Portfolio Daily Return %"},inplace = True )

In [61]:
portfolio_B_daily_returns.head()

,Date,Portfolio Daily Return %
0,2025-12-26,0.0000
1,2025-12-29,-1.2020
2,2025-12-30,-0.3905
3,2025-12-31,-0.7000
4,2026-01-02,-0.7990


In [62]:
initial_capital = investor_profiles["Investor_B"]["capital"]

portfolio_B_daily_returns["Portfolio Value"] = round((initial_capital * 
                                                (1 + portfolio_B_daily_returns["Portfolio Daily Return %"] / 100).cumprod()),3)
portfolio_B_daily_returns.head()

,Date,Portfolio Daily Return %,Portfolio Value
0,2025-12-26,0.0000,10000.000
1,2025-12-29,-1.2020,9879.800
2,2025-12-30,-0.3905,9841.219
3,2025-12-31,-0.7000,9772.331
4,2026-01-02,-0.7990,9694.250


In [63]:
portfolio_B_daily_returns["Portfolio Cumulative Return %"] = round(((portfolio_B_daily_returns["Portfolio Value"] - initial_capital) / initial_capital )*100,3)
portfolio_B_daily_returns.head(3)

,Date,Portfolio Daily Return %,Portfolio Value,Portfolio Cumulative Return %
0,2025-12-26,0.0000,10000.000,0.000
1,2025-12-29,-1.2020,9879.800,-1.202
2,2025-12-30,-0.3905,9841.219,-1.588


In [64]:
portfolio_A_daily_returns.iloc[-1]

Date                             2026-06-25 00:00:00
Portfolio Daily Return %                       1.414
Portfolio Value                            10841.796
Portfolio Cumulative Return %                  8.418
Name: 123, dtype: object

In [65]:
portfolio_B_daily_returns.iloc[-1]

Date                             2026-06-25 00:00:00
Portfolio Daily Return %                     -2.5185
Portfolio Value                             9600.562
Portfolio Cumulative Return %                 -3.994
Name: 123, dtype: object

In [66]:
return_A = portfolio_A_daily_returns["Portfolio Cumulative Return %"].iloc[-1]
return_B = portfolio_B_daily_returns["Portfolio Cumulative Return %"].iloc[-1]

In [67]:
# expected annual return is

annual_return_A = round((portfolio_A_daily_returns["Portfolio Daily Return %"].mean()* 252),2)
annual_return_B = round((portfolio_B_daily_returns["Portfolio Daily Return %"].mean()* 252),2)
print(annual_return_A)
print(annual_return_B)

17.67
-5.2


#### calculate risk to check which portfolio is more volatile

In [68]:
portfolio_A_daily_returns["Portfolio Daily Return %"].std()

0.9964526252903051

In [69]:
portfolio_B_daily_returns["Portfolio Daily Return %"].std()

1.567983901759902

In [70]:
annual_risk_A = (portfolio_A_daily_returns["Portfolio Daily Return %"].std() * (252 ** 0.5))

annual_risk_B = (portfolio_B_daily_returns["Portfolio Daily Return %"].std() * (252 ** 0.5))

print(round(annual_risk_A,2))
print(round(annual_risk_B,2))

15.82
24.89


In [71]:
# assuming risk-free rate as 0 
sharpe_A = round(annual_return_A / annual_risk_A,2)
sharpe_B = round(annual_return_B / annual_risk_B,2)
print(sharpe_A)
print(sharpe_B)

1.12
-0.21


In [72]:
# creating the overall summary
portfolio_summary = pd.DataFrame({
                                "Portfolio": ["Conservative", "Growth"],
                                "Capital": [10000, 10000],
                                "Current Value": [
                                        round(portfolio_A_daily_returns["Portfolio Value"].iloc[-1], 2),
                                        round(portfolio_B_daily_returns["Portfolio Value"].iloc[-1], 2)],
                                "Total Return %": [
                                        round(portfolio_A_daily_returns["Portfolio Cumulative Return %"].iloc[-1], 2),
                                        round(portfolio_B_daily_returns["Portfolio Cumulative Return %"].iloc[-1], 2)],
                                "Annual Return %": [
                                        annual_return_A,
                                        annual_return_B],
                                "Annual Risk %": [
                                        round(annual_risk_A, 2),
                                        round(annual_risk_B, 2)],
                                "Sharpe Ratio": [
                                        sharpe_A,
                                        sharpe_B]
                                })

In [73]:
portfolio_summary.head()

,Portfolio,Capital,Current Value,Total Return %,Annual Return %,Annual Risk %,Sharpe Ratio
0,Conservative,10000,10841.80,8.42,17.67,15.82,1.12
1,Growth,10000,9600.56,-3.99,-5.20,24.89,-0.21


In [74]:
portfolio_A_daily_returns["Portfolio"] = "Conservative"

In [75]:
portfolio_B_daily_returns["Portfolio"] = "Growth"

In [76]:
portfolio_daily_returns = pd.concat([portfolio_A_daily_returns, portfolio_B_daily_returns],ignore_index=True)

In [77]:
portfolio_daily_returns = portfolio_daily_returns[["Date","Portfolio","Portfolio Daily Return %","Portfolio Value",
                                                   "Portfolio Cumulative Return %"]]

In [78]:
portfolio_daily_returns = portfolio_daily_returns.sort_values(["Date", "Portfolio"])

In [79]:
portfolio_daily_returns = portfolio_daily_returns.reset_index(drop=True)

In [80]:
portfolio_daily_returns.head()

,Date,Portfolio,Portfolio Daily Return %,Portfolio Value,Portfolio Cumulative Return %
0,2025-12-26,Conservative,0.000,10000.00,0.000
1,2025-12-26,Growth,0.000,10000.00,0.000
2,2025-12-29,Conservative,-0.751,9924.90,-0.751
3,2025-12-29,Growth,-1.202,9879.80,-1.202
4,2025-12-30,Conservative,-0.227,9902.37,-0.976


In [81]:
historical_stock_data = historical_stock_data.sort_values(["Ticker", "Date"])

In [82]:
historical_stock_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
0,AAPL,Technology,2025-12-26,272.893005,274.859353,272.353998,273.651606,21521800,NaN
20,AAPL,Technology,2025-12-29,273.252350,273.851213,271.844961,272.184327,23715200,0.13
40,AAPL,Technology,2025-12-30,272.573547,273.571693,271.775043,272.304059,22139600,-0.25
60,AAPL,Technology,2025-12-31,271.355835,273.172467,271.246054,272.553622,27293600,-0.45
80,AAPL,Technology,2026-01-02,270.507446,277.324767,268.501164,271.755128,37838100,-0.31


In [83]:
print(historical_stock_data.shape)
print(portfolio_daily_returns.shape)
print(portfolio_summary.shape)

(2480, 9)
(248, 5)
(2, 7)


In [84]:
historical_stock_data = historical_stock_data.reset_index(drop=True)
portfolio_daily_returns = portfolio_daily_returns.reset_index(drop=True)
portfolio_summary = portfolio_summary.reset_index(drop=True)

In [85]:
historical_stock_data.to_csv("historical_stock_data.csv", index=False)
portfolio_daily_returns.to_csv("portfolio_daily_returns.csv", index=False)
portfolio_summary.to_csv("portfolio_summary.csv", index=False)
stock_performance_summary.to_csv("stock_performance_summary.csv",index=False)

### generate PDF report

In [86]:
import pandas as pd

In [87]:
portfolio_summary = pd.read_csv("portfolio_summary.csv")

portfolio_summary

,Portfolio,Capital,Current Value,Total Return %,Annual Return %,Annual Risk %,Sharpe Ratio
0,Conservative,10000,10841.80,8.42,17.67,15.82,1.12
1,Growth,10000,9600.56,-3.99,-5.20,24.89,-0.21


In [88]:
max_return  = portfolio_summary["Total Return %"].max()
max_return

8.42

In [89]:
best_portfolio = portfolio_summary.loc[portfolio_summary["Total Return %"]==max_return,"Portfolio"].iloc[0]
worst_portfolio = portfolio_summary.loc[portfolio_summary["Total Return %"] == portfolio_summary["Total Return %"].min(),"Portfolio"].iloc[0]

In [90]:
print(best_portfolio)
print(worst_portfolio)

Conservative
Growth


In [91]:
return_difference = round(portfolio_summary["Total Return %"].max() - portfolio_summary["Total Return %"].min(),2) 

In [92]:
return_difference

12.41

In [93]:
print(f"Best Performing Portfolio: {best_portfolio}")
print(f"{best_portfolio} Portfolio Outperformed {worst_portfolio} Portfolio by {return_difference}%")

Best Performing Portfolio: Conservative
Conservative Portfolio Outperformed Growth Portfolio by 12.41%


In [94]:
portfolio_daily_returns = pd.read_csv("portfolio_daily_returns.csv")

In [95]:
report_date = portfolio_daily_returns["Date"].max()
report_date

'2026-06-25'

In [96]:
report_text = f"""
📊 Portfolio Report

Report Date: {report_date}

"""

In [97]:
print(report_text)


📊 Portfolio Report

Report Date: 2026-06-25




In [98]:
for _, portfolio in portfolio_summary.iterrows():

    report_text += f"""

Portfolio: {portfolio["Portfolio"]}

Current Value: ₹{portfolio["Current Value"]}

Total Return: {portfolio["Total Return %"]}%

Annual Return: {portfolio["Annual Return %"]}%

Annual Risk: {portfolio["Annual Risk %"]}%

Sharpe Ratio: {portfolio["Sharpe Ratio"]}

"""

In [99]:
print(report_text)


📊 Portfolio Report

Report Date: 2026-06-25



Portfolio: Conservative

Current Value: ₹10841.8

Total Return: 8.42%

Annual Return: 17.67%

Annual Risk: 15.82%

Sharpe Ratio: 1.12



Portfolio: Growth

Current Value: ₹9600.56

Total Return: -3.99%

Annual Return: -5.2%

Annual Risk: 24.89%

Sharpe Ratio: -0.21




In [100]:
report_text += f"""

Portfolio Comparison

Best Performing Portfolio: {best_portfolio}

{best_portfolio} Portfolio Outperformed Conservative Portfolio by {return_difference}%

"""

In [101]:
print(report_text)


📊 Portfolio Report

Report Date: 2026-06-25



Portfolio: Conservative

Current Value: ₹10841.8

Total Return: 8.42%

Annual Return: 17.67%

Annual Risk: 15.82%

Sharpe Ratio: 1.12



Portfolio: Growth

Current Value: ₹9600.56

Total Return: -3.99%

Annual Return: -5.2%

Annual Risk: 24.89%

Sharpe Ratio: -0.21



Portfolio Comparison

Best Performing Portfolio: Conservative

Conservative Portfolio Outperformed Conservative Portfolio by 12.41%




!pip install reportlab

In [102]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

In [103]:
# creating a variable to save the path for report on my system
pdf_path = f"reports/portfolio_report_{report_date}.pdf"

In [104]:
# create an empty pdf document object 
doc = SimpleDocTemplate(pdf_path)

In [105]:
# get default text formatting styles
styles = getSampleStyleSheet()

In [106]:
elements = []

In [107]:
# add Title and recent date
elements.append(Paragraph("Portfolio Performance Report", styles["Title"]))

elements.append(Paragraph(f"Report Date: {report_date}", styles["Normal"]))

elements.append(Spacer(1, 12))

In [108]:
# add portfolio details of each
for _, portfolio in portfolio_summary.iterrows():

    elements.append(Paragraph(f"<b>Portfolio:</b> {portfolio['Portfolio']}",styles["Heading2"]))

    elements.append(Paragraph(f"Current Value: ₹{portfolio['Current Value']}",styles["Normal"]))

    elements.append(Paragraph(f"Total Return: {portfolio['Total Return %']}%",styles["Normal"]))

    elements.append(Paragraph(f"Annual Return: {portfolio['Annual Return %']}%",styles["Normal"]))

    elements.append(Paragraph(f"Annual Risk: {portfolio['Annual Risk %']}%",styles["Normal"]))

    elements.append(Paragraph(f"Sharpe Ratio: {portfolio['Sharpe Ratio']}",styles["Normal"]))

    elements.append(Spacer(1, 12))

In [109]:
# add the comparision section
elements.append(Paragraph("Portfolio Comparison", styles["Heading1"]))

elements.append(Paragraph(f"Best Performing Portfolio: {best_portfolio}",styles["Normal"]))

elements.append(Paragraph(f"{best_portfolio} Portfolio Outperformed Conservative Portfolio by {return_difference}%",styles["Normal"]))

In [110]:
print(len(elements))

20


In [111]:
doc.build(elements)

print("PDF Report Created Successfully!")

PDF Report Created Successfully!
